In [ ]:
import sys
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import poisson


# Defining global parameters for the problem
class GbikeRental:
    @staticmethod
    def max_bikes():
        return 20  # Maximum number of bikes at each location

    @staticmethod
    def gamma():
        return 0.9  # Discount factor for future rewards

    @staticmethod
    def credit_reward():
        return 10  # Reward per rented bike

    @staticmethod
    def moving_cost(action):
        # First bike is free when moving from A -> B
        return max(0, abs(action) - 1) * 2

    @staticmethod
    def parking_cost(bikes):
        # Additional parking cost for more than 10 bikes
        return 4 if bikes > 10 else 0


# Class for Poisson distribution with pre-computed probabilities
class Poisson:
    def __init__(self, lam):
        self.lam = lam
        self.epsilon = 0.01  # Minimum probability threshold
        self.vals = {}
        self.alpha = 0
        self.beta = 0
        self._compute_distribution()

    def _compute_distribution(self):
        total_prob = 0
        for n in range(100):  # Arbitrary large range to compute probabilities
            prob = poisson.pmf(n, self.lam)
            if prob < self.epsilon and total_prob > 1 - self.epsilon:
                break
            self.vals[n] = prob
            total_prob += prob
            self.beta = n
        # Normalize probabilities
        missing_prob = (1 - total_prob) / (self.beta - self.alpha + 1)
        for n in self.vals:
            self.vals[n] += missing_prob

    def f(self, n):
        return self.vals.get(n, 0)


# Class to define location properties
class Location:
    def __init__(self, req_rate, ret_rate):
        self.req_rate = req_rate  # Lambda for requests
        self.ret_rate = ret_rate  # Lambda for returns
        self.req_poisson = Poisson(req_rate)
        self.ret_poisson = Poisson(ret_rate)


# Apply an action to a state
def apply_action(state, action):
    bikes_a = max(0, min(state[0] - action, GbikeRental.max_bikes()))
    bikes_b = max(0, min(state[1] + action, GbikeRental.max_bikes()))
    return [bikes_a, bikes_b]


# Calculate expected reward for a state and action
def expected_reward(state, action):
    global value

    # Apply action and compute immediate cost
    new_state = apply_action(state, action)
    cost = GbikeRental.moving_cost(action)

    # Initialize expected value
    expected_val = -cost

    # Iterate over all possible requests and returns at both locations
    for req_a in A.req_poisson.vals:
        for req_b in B.req_poisson.vals:
            for ret_a in A.ret_poisson.vals:
                for ret_b in B.ret_poisson.vals:
                    # Probability of this combination
                    prob = (
                        A.req_poisson.f(req_a)
                        * B.req_poisson.f(req_b)
                        * A.ret_poisson.f(ret_a)
                        * B.ret_poisson.f(ret_b)
                    )

                    # Fulfilled requests
                    rented_a = min(req_a, new_state[0])
                    rented_b = min(req_b, new_state[1])

                    # Calculate rewards
                    reward = (rented_a + rented_b) * GbikeRental.credit_reward()

                    # Bikes left after requests, and returned bikes
                    remaining_a = max(0, new_state[0] - rented_a + ret_a)
                    remaining_b = max(0, new_state[1] - rented_b + ret_b)

                    # Apply parking cost
                    reward -= GbikeRental.parking_cost(remaining_a)
                    reward -= GbikeRental.parking_cost(remaining_b)

                    # Ensure remaining bikes do not exceed max capacity
                    remaining_a = min(remaining_a, GbikeRental.max_bikes())
                    remaining_b = min(remaining_b, GbikeRental.max_bikes())

                    # Add to expected value using Bellman's equation
                    expected_val += prob * (reward + GbikeRental.gamma() * value[remaining_a][remaining_b])

    return expected_val


# Policy evaluation function
def policy_evaluation():
    global value
    epsilon = policy_evaluation.epsilon
    policy_evaluation.epsilon /= 10

    while True:
        delta = 0
        for i in range(value.shape[0]):
            for j in range(value.shape[1]):
                old_value = value[i][j]
                value[i][j] = expected_reward([i, j], policy[i][j])
                delta = max(delta, abs(value[i][j] - old_value))
                print('.', end='', flush=True)

        print(f" Delta: {delta}")
        if delta < epsilon:
            break


# Policy improvement function
def policy_improvement():
    global policy
    policy_stable = True

    for i in range(policy.shape[0]):
        for j in range(policy.shape[1]):  # Corrected this line
            old_action = policy[i][j]
            max_action_value = float('-inf')
            best_action = None

            # Determine feasible actions
            max_transfer_a_to_b = min(i, 5)  # Maximum bikes transferable from A to B
            max_transfer_b_to_a = -min(j, 5)  # Maximum bikes transferable from B to A

            # Evaluate each action in the feasible range
            for action in range(max_transfer_b_to_a, max_transfer_a_to_b + 1):
                action_value = expected_reward([i, j], action)

                if action_value > max_action_value:
                    max_action_value = action_value
                    best_action = action

            # Update policy
            policy[i][j] = best_action
            if old_action != best_action:
                policy_stable = False

    return policy_stable


# Save policy heatmap
def save_policy():
    save_policy.counter += 1
    ax = sns.heatmap(policy, linewidth=0.5, cmap="Greens")
    ax.invert_yaxis()
    plt.title(f"Policy {save_policy.counter}")
    plt.savefig(f"policy_{save_policy.counter}.svg")
    plt.close()


# Save value heatmap
def save_value():
    save_value.counter += 1
    ax = sns.heatmap(value, linewidth=0.5, cmap="Blues")
    ax.invert_yaxis()
    plt.title(f"Value {save_value.counter}")
    plt.savefig(f"value_{save_value.counter}.svg")
    plt.close()


# Initialize locations
A = Location(3, 3)  # Requests and returns for location A
B = Location(4, 2)  # Requests and returns for location B

# Initialize matrices for values and policies
value = np.zeros((GbikeRental.max_bikes() + 1, GbikeRental.max_bikes() + 1))
policy = np.zeros_like(value, dtype=int)

# Initialize static variables
policy_evaluation.epsilon = 50
save_policy.counter = 0
save_value.counter = 0

# Policy iteration loop
while True:
    policy_evaluation()
    is_policy_stable = policy_improvement()
    save_value()
    save_policy()

    if is_policy_stable:
        break

......................................................................................................................................................................................................................................................................................................................................................................................................................................................... Delta: 171.89024673373066
......................................................................................................................................................................................................................................................................................................................................................................................................................................................... Delta: 120.98703795723475
................................................................